In [ ]:
import os
import numpy as np
import pickle
import torch
from argparse import ArgumentParser
from tqdm import tqdm
import glob

from articulate.model import ParametricModel
from articulate import math
from config import paths, datasets
import os
import math
import numpy as np
import torch
torch.set_printoptions(sci_mode=False)
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import lightning as L
from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch import seed_everything
from argparse import ArgumentParser
from pathlib import Path
from typing import List
from tqdm import tqdm 
import wandb

from constants import MODULES
from data import PoseDataModule
from utils.file_utils import (
    get_datestring, 
    make_dir, 
    get_dir_number, 
    get_best_checkpoint
)
from config import paths, train_hypers, finetune_hypers





In [3]:

# set precision for Tensor cores
torch.set_float32_matmul_precision('medium')


class TrainingManager:
    """Manage training of MobilePoser modules."""
    def __init__(self, finetune: str=None, fast_dev_run: bool=False):
        self.finetune = finetune
        self.fast_dev_run = fast_dev_run
        self.hypers = finetune_hypers if finetune else train_hypers

    def _setup_wandb_logger(self, save_path: Path):
        wandb_logger = WandbLogger(
            project=save_path.name, 
            name=get_datestring(),
            save_dir=save_path
        ) 
        return wandb_logger

    def _setup_callbacks(self, save_path):
        checkpoint_callback = ModelCheckpoint(
                monitor="validation_step_loss",
                save_top_k=3,
                mode="min",
                verbose=False,
                dirpath=save_path, 
                save_weights_only=True,
                filename="{epoch}-{validation_step_loss:.4f}"
                )
        return checkpoint_callback

    def _setup_trainer(self, module_path: Path):
        print("Module Path: ", module_path.name, module_path)
        logger = self._setup_wandb_logger(module_path) 
        checkpoint_callback = self._setup_callbacks(module_path)
        trainer = L.Trainer(
                fast_dev_run=self.fast_dev_run,
                min_epochs=self.hypers.num_epochs,
                max_epochs=self.hypers.num_epochs,
                devices=[self.hypers.device], 
                accelerator=self.hypers.accelerator,
                logger=logger,
                callbacks=[checkpoint_callback],
                deterministic=True
                )
        return trainer

    def train_module(self, model: L.LightningModule, module_name: str, checkpoint_path: Path):
        # set the appropriate hyperparameters
        model.hypers = self.hypers 

        # create directory for module
        module_path = checkpoint_path / module_name
        make_dir(module_path)
        datamodule = PoseDataModule(finetune=self.finetune)
        trainer = self._setup_trainer(module_path)

        print()
        print("-" * 50)
        print(f"Training Module: {module_name}")
        print("-" * 50)
        print()

        try:
            trainer.fit(model, datamodule=datamodule)
        finally:
            wandb.finish()
            del model
            torch.cuda.empty_cache()


def get_checkpoint_path(finetune: str, init_from: str):
    if finetune:
        # finetune from a checkpoint
        parts = init_from.split(os.path.sep)
        checkpoint_path = Path(parts[0]) / parts[1]
        finetune_dir = f"finetuned_{finetune}"
        checkpoint_path = checkpoint_path / finetune_dir
    else:
        # make directory for trained models
        dir_name = get_dir_number(paths.checkpoint)
        checkpoint_path = paths.checkpoint / str(dir_name)

    # create directory if it does not exist
    checkpoint_path.mkdir(parents=True, exist_ok=True)

    return checkpoint_path


In [4]:
# python train.py --module joints --init-from checkpoints/$2/joints --finetune dip
#     python train.py --module poser --init-from checkpoints/$2/poser --finetune dip

In [13]:
seed_everything(42, workers=True)

paths.checkpoint.mkdir(exist_ok=True)
finetune = "dip"
module = 'joints'
init_from =  r"C:\deepika\2.mobileposer\MobilePoser\mobileposer\checkpoints\5\joints"
fast_dev_run = False


checkpoint_path = get_checkpoint_path(finetune, init_from)
training_manager = TrainingManager(
    finetune=finetune,
    fast_dev_run=fast_dev_run
)

if module:
    if module not in MODULES:
        raise ValueError(f"Module {module} not found.")

    module_name = module              # "poser"
    module_cls = MODULES[module_name] # models.poser.Poser (a class)

    model_dir = Path(init_from)
    model = module_cls()              # init model from scratch

    if finetune:
        print('finetuning sista')
        model_path = get_best_checkpoint(model_dir)
        model = module_cls.from_pretrained(
            model_path=os.path.join(model_dir, model_path)
        )

    training_manager.train_module(model, module_name, checkpoint_path)
else:
    # train all modules
    for module_name, module_cls in MODULES.items():
        training_manager.train_module(module_cls(), module_name, checkpoint_path)


Seed set to 42
GPU available: True (cuda), used: True


finetuning sista
Module Path:  joints C:deepika\finetuned_dip\joints


TPU available: False, using: 0 TPU cores
wandb: WARNING `resume` will be ignored since W&B syncing is set to `offline`. Starting a new run with run id d5d4kjct.



--------------------------------------------------
Training Module: joints
--------------------------------------------------



100%|██████████| 1/1 [00:00<00:00,  1.57it/s]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name   | Type    | Params | Mode  | FLOPs
---------------------------------------------------
0 | joints | RNN     | 2.7 M  | train | 0    
1 | loss   | MSELoss | 0      | train | 0    
---------------------------------------------------
2.7 M     Trainable params
0         Non-trainable params
2.7 M     Total params
10.729    Total estimated model params size (MB)
6         Modules in train mode
0         Modules in eval mode
0         Total Flops


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=15` reached.


epoch,▁▁▁▁▂▂▂▃▃▃▃▃▃▃▃▃▃▃▃▄▅▅▅▅▅▅▅▅▅▅▆▇▇▇▇▇▇▇██
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,█▅▅▄▃▃▃▂▂▂▂▁▁▁▁
trainer/global_step,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇████
training_step_loss,▇█▄▅▄▄▇▄▂▃▄▄▃▂▂▃▅▂▄▃▃▂▃▃▂▂▂▃▁▄▂▂▂▄▁▃▂▂▂▁
val_loss,█▆▅▅▄▃▃▃▂▂▂▁▁▁▁
validation_step_loss,█▆▅▅▄▃▃▃▂▂▂▁▁▁▁
epoch,14
learning_rate,5e-05
train_loss,0.00272
trainer/global_step,5219


In [6]:
import os
from config import paths

print("CWD:", os.getcwd())
print("processed_datasets:", paths.processed_datasets)
print("exists:", os.path.exists(paths.processed_datasets))
print("dip_test expected:", os.path.join(paths.processed_datasets, "eval", "dip_test.pt"))
print("dip_test exists:", os.path.exists(os.path.join(paths.processed_datasets, "eval", "dip_test.pt")))


CWD: c:\deepika\2.mobileposer\MobilePoser\mobileposer
processed_datasets: c:\deepika\2.mobileposer\MobilePoser\mobileposer\data\processed_datasets
exists: True
dip_test expected: c:\deepika\2.mobileposer\MobilePoser\mobileposer\data\processed_datasets\eval\dip_test.pt
dip_test exists: True


In [7]:
import torch, os
from config import paths

p = os.path.join(paths.processed_datasets, "eval", "dip_test.pt")
d = torch.load(p, map_location="cpu")

print("keys:", d.keys())
print("len(acc):", len(d["acc"]))
print("len(ori):", len(d["ori"]))
print("len(pose):", len(d["pose"]))
print("len(tran):", len(d["tran"]))


keys: dict_keys(['joint', 'pose', 'shape', 'tran', 'acc', 'ori'])
len(acc): 19
len(ori): 19
len(pose): 0
len(tran): 19


C:\Users\degu03\AppData\Local\Temp\ipykernel_22692\2914679804.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(p, map_location="cpu")


In [9]:
import os, torch
from config import paths

p = os.path.join(paths.processed_datasets, "dip_train.pt")
d = torch.load(p, map_location="cpu")
print(len(d["acc"]), len(d["pose"]))


41 0


C:\Users\degu03\AppData\Local\Temp\ipykernel_22692\2122012215.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(p, map_location="cpu")


In [10]:
from config import paths
import os

print("paths.eval_dir:", paths.eval_dir)
print("expected dip_test:", paths.eval_dir / "dip_test.pt")
print("exists:", os.path.exists(paths.eval_dir / "dip_test.pt"))


paths.eval_dir: c:\deepika\2.mobileposer\MobilePoser\mobileposer\data\processed_datasets\eval
expected dip_test: c:\deepika\2.mobileposer\MobilePoser\mobileposer\data\processed_datasets\eval\dip_test.pt
exists: True


In [11]:
p = os.path.join(paths.processed_datasets, "eval", "dip_test.pt")
print("loaded file:", p)


loaded file: c:\deepika\2.mobileposer\MobilePoser\mobileposer\data\processed_datasets\eval\dip_test.pt
